# Per-Layer Adaptive Softmax Training

Fine-tune **gemma-3n-e2b** with per-layer trainable polynomial parameters for adaptive temperature softmax.

Based on "Softmax is Not Enough" paper - trains unique polynomial coefficients for each transformer layer.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gulkoa/softmax-is-meh/blob/main/train_adaptive_colab.ipynb)

## 1. Installation

**IMPORTANT**: Run this cell first, then **restart the runtime** before running other cells.

In [ ]:
%%capture
# Use unsloth's official Colab installation (from their working notebooks)
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
# Clone CLRS repo for algorithmic reasoning dataset
!git clone https://github.com/google-deepmind/clrs.git clrs-repo 2>/dev/null || echo "Already cloned"
import sys
sys.path.append('clrs-repo')

## 2. Adaptive Softmax Module

In [ ]:
import torch
from torch import nn

class AdaptivePolynomialConfig(nn.Module):
    """Manages per-layer polynomial parameters for adaptive temperature softmax."""
    
    def __init__(self, num_layers: int, poly_degree: int = 5):
        super().__init__()
        self.num_layers = num_layers
        self.poly_degree = poly_degree
        
        # Paper's initial values from Figure 6
        init_values = torch.tensor([-0.037, 0.481, -2.3, 4.917, -1.791], dtype=torch.float32)
        
        # Shape: [num_layers, poly_degree]
        self.poly_coeffs = nn.Parameter(
            init_values.unsqueeze(0).expand(num_layers, -1).clone()
        )
    
    def get_layer_coeffs(self, layer_idx: int) -> torch.Tensor:
        return self.poly_coeffs[layer_idx]
    
    def save_checkpoint(self, path: str):
        torch.save({
            'poly_coeffs': self.poly_coeffs.data,
            'num_layers': self.num_layers,
            'poly_degree': self.poly_degree,
        }, path)
    
    @classmethod
    def load_checkpoint(cls, path: str) -> 'AdaptivePolynomialConfig':
        checkpoint = torch.load(path)
        config = cls(
            num_layers=checkpoint['num_layers'],
            poly_degree=checkpoint['poly_degree']
        )
        config.poly_coeffs.data = checkpoint['poly_coeffs']
        return config


def get_polynomial_value(x: torch.Tensor, coeffs: torch.Tensor) -> torch.Tensor:
    """Evaluate polynomial using Horner's method."""
    result = torch.zeros_like(x)
    for i in range(len(coeffs) - 1):
        result = (result + coeffs[i]) * x
    return result + coeffs[-1]


def softmax_adaptive_temperature(logits, dim, poly_coeffs, dtype=torch.float32):
    """Adaptive temperature softmax with proper device handling."""
    original_probs = torch.softmax(logits, dim=dim, dtype=dtype)
    entropy = torch.sum(
        -original_probs * torch.log(original_probs + 1e-9),
        dim=-1, keepdim=True
    )
    
    poly_val = get_polynomial_value(entropy, poly_coeffs)
    
    beta = torch.where(
        entropy > 0.5,
        torch.maximum(poly_val, torch.ones_like(poly_val)),
        torch.ones_like(entropy)
    )
    
    return torch.softmax(logits * beta, dim=dim, dtype=dtype)

print("Adaptive softmax module loaded!")

## 3. Configuration

In [ ]:
# Model
MODEL_NAME = "google/gemma-3n-e2b"
NUM_LAYERS = 34  # gemma-3n-e2b has 34 transformer layers
MAX_SEQ_LENGTH = 1024

# Training
MAX_STEPS = 200
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
LR_LORA = 2e-4
LR_POLY = 2e-3  # 10x higher for polynomial params
SAVE_STEPS = 50
LOGGING_STEPS = 10

# Output
OUTPUT_DIR = "/content/results_adaptive"
CHECKPOINT_DIR = f"{OUTPUT_DIR}/checkpoints"

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")

## 4. Load Model with Unsloth

In [ ]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

In [ ]:
# Apply LoRA
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=8,
    lora_alpha=8,
    lora_dropout=0,
)

## 5. Create Per-Layer Polynomial Config

In [ ]:
poly_config = AdaptivePolynomialConfig(num_layers=NUM_LAYERS)
poly_config = poly_config.to(model.device)

print(f"Polynomial parameters shape: {poly_config.poly_coeffs.shape}")
print(f"Total trainable poly params: {poly_config.poly_coeffs.numel()}")

## 6. Patch Attention with Adaptive Softmax

In [ ]:
import torch.nn.functional as F

# Store reference globally
_GLOBAL_POLY_CONFIG = poly_config
_CURRENT_LAYER_IDX = [0]
_original_softmax = F.softmax
_in_attention = [False]

def patched_softmax(input, dim=None, _stacklevel=3, dtype=None):
    global _GLOBAL_POLY_CONFIG, _CURRENT_LAYER_IDX
    
    if _in_attention[0] and input.dim() == 4 and dim == -1:
        layer_idx = min(_CURRENT_LAYER_IDX[0], NUM_LAYERS - 1)
        coeffs = _GLOBAL_POLY_CONFIG.get_layer_coeffs(layer_idx)
        result = softmax_adaptive_temperature(input, dim, coeffs, dtype=dtype or torch.float32)
        return result.to(input.dtype) if dtype is None else result
    
    if dtype is not None:
        return _original_softmax(input, dim=dim, dtype=dtype)
    return _original_softmax(input, dim=dim)

F.softmax = patched_softmax

# Wrap attention layers
def wrap_attention_layer(layer_module, layer_idx):
    original_forward = layer_module.forward
    
    def wrapped_forward(*args, **kwargs):
        _CURRENT_LAYER_IDX[0] = layer_idx
        _in_attention[0] = True
        try:
            result = original_forward(*args, **kwargs)
        finally:
            _in_attention[0] = False
        return result
    
    layer_module.forward = wrapped_forward

# Find and wrap layers
if hasattr(model, 'model'):
    base_model = model.model
    if hasattr(base_model, 'model'):
        base_model = base_model.model
    if hasattr(base_model, 'layers'):
        layers = base_model.layers
    elif hasattr(base_model, 'language_model') and hasattr(base_model.language_model, 'model'):
        layers = base_model.language_model.model.layers
    else:
        layers = []
else:
    layers = []

for idx, layer in enumerate(layers):
    if hasattr(layer, 'self_attn'):
        wrap_attention_layer(layer.self_attn, idx)

print(f"Wrapped {len(layers)} attention layers with adaptive softmax")

## 7. Setup CLRS Dataset

In [ ]:
from datasets import IterableDataset, Features, Value, Dataset
from clrs._src.clrs_text.huggingface_generators import clrs_generator
from unsloth.chat_templates import get_chat_template

algos_and_lengths = {
    "minimum": [8, 16, 24, 32],
}

ds = IterableDataset.from_generator(
    clrs_generator,
    features=Features({
        "text": Value("string"),
        "question": Value("string"),
        "answer": Value("string"),
        "algo_name": Value("string"),
        "length": Value("int32"),
        "use_hints": Value("bool_"),
    }),
    gen_kwargs={"algos_and_lengths": algos_and_lengths},
)

tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")

def formatting_func(examples):
    texts = []
    for q, a in zip(examples['question'], examples['answer']):
        convo = [
            {"role": "user", "content": f"Find the index of the minimum value.\n{q}"},
            {"role": "assistant", "content": str(a).strip()},
        ]
        text = tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        if text.startswith('<bos>'):
            text = text[5:]
        texts.append(text)
    return {"text": texts}

# Create finite dataset
print("Generating training samples...")
train_samples = []
sample_iter = iter(ds)
for _ in range(1000):
    try:
        train_samples.append(next(sample_iter))
    except StopIteration:
        break

train_dataset = Dataset.from_list(train_samples)
train_dataset = train_dataset.map(formatting_func, batched=True, remove_columns=train_dataset.column_names)
print(f"Training samples: {len(train_dataset)}")

## 8. Training

In [ ]:
from torch.optim import AdamW
from trl import SFTTrainer, SFTConfig

# Custom optimizer with separate param groups
model_params = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW([
    {"params": model_params, "lr": LR_LORA},
    {"params": poly_config.parameters(), "lr": LR_POLY},
])

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    warmup_steps=10,
    max_steps=MAX_STEPS,
    learning_rate=LR_LORA,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=5,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=training_args,
    optimizers=(optimizer, None),
)

In [ ]:
# Add polynomial checkpoint hook
original_training_step = trainer.training_step

def custom_training_step(model, inputs):
    result = original_training_step(model, inputs)
    
    if trainer.state.global_step > 0 and trainer.state.global_step % SAVE_STEPS == 0:
        poly_path = f"{CHECKPOINT_DIR}/poly_coeffs_step_{trainer.state.global_step}.pt"
        poly_config.save_checkpoint(poly_path)
        print(f"\n  Saved polynomial checkpoint: {poly_path}")
    
    return result

trainer.training_step = custom_training_step

In [ ]:
print("="*60)
print("Starting training!")
print("="*60)

trainer.train()

## 9. Save Results

In [ ]:
# Save final polynomial coefficients
final_poly_path = f"{OUTPUT_DIR}/poly_coeffs_final.pt"
poly_config.save_checkpoint(final_poly_path)
print(f"Saved final polynomial coefficients: {final_poly_path}")

# Save model
model.save_pretrained(f"{OUTPUT_DIR}/model_final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/model_final")
print(f"Saved model: {OUTPUT_DIR}/model_final")

In [ ]:
# Print final coefficients
print("\nFinal polynomial coefficients per layer:")
print("-" * 50)
for i in range(min(NUM_LAYERS, 10)):
    coeffs = poly_config.poly_coeffs[i].detach().cpu().tolist()
    print(f"  Layer {i:2d}: [{', '.join(f'{c:.4f}' for c in coeffs)}]")
if NUM_LAYERS > 10:
    print(f"  ... ({NUM_LAYERS - 10} more layers)")

In [ ]:
# Visualize coefficient evolution
import matplotlib.pyplot as plt

coeffs_np = poly_config.poly_coeffs.detach().cpu().numpy()

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
coeff_names = ['a4 (x^4)', 'a3 (x^3)', 'a2 (x^2)', 'a1 (x)', 'a0']

for i, ax in enumerate(axes):
    ax.bar(range(NUM_LAYERS), coeffs_np[:, i])
    ax.set_title(coeff_names[i])
    ax.set_xlabel('Layer')
    ax.axhline(y=coeffs_np[0, i], color='r', linestyle='--', alpha=0.5, label='Initial')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/coefficients_per_layer.png")
plt.show()

print(f"\nSaved visualization: {OUTPUT_DIR}/coefficients_per_layer.png")

In [ ]:
# Download checkpoints (Colab)
!zip -r /content/adaptive_results.zip {OUTPUT_DIR}

from google.colab import files
files.download('/content/adaptive_results.zip')